In [1]:
# we use the dataset https://github.com/gaia-solutions-on-demand/DFireDataset/blob/master/figures/dfire_examples.png as a training set
# we also annotated manually using roboflow part of the dataset from "defi1" to serve as a test set 

# Import

In [1]:
! pip install ultralytics
! pip install ruamel.yaml
! pip install captum

In [2]:
import os
import random
import argparse

import numpy as np
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.v2 as transforms

from captum.attr import LayerGradCam

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from ultralytics import YOLO
from ultralytics.utils import DEFAULT_CFG_DICT
from ruamel.yaml import YAML

yaml = YAML(typ='safe')
with open('configs/train.yaml', 'r') as f:
    cfg = yaml.load(f)

print(f"Configured dataset fraction: {cfg['fraction']:.0%}")
print(f"Ultralytics default fraction: {DEFAULT_CFG_DICT['fraction']}")

/home/hackia_group/hackia/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configured dataset fraction: 100%
Ultralytics default fraction: 1.0


# Utils

In [3]:
def set_seed(seed: int = 42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def load_yaml(path):
    yaml = YAML(typ='safe')
    with open(path, 'r') as f:
        return yaml.load(f)
    
config = "configs/train.yaml"
seed=42
device="cuda"

# Bounding box

## Images

In [6]:
from ultralytics import YOLO

# Load your fine-tuned model
model = YOLO("/home/hackia_group/Phase1/fire-detection/yolo26_fire_detection7/weights/best.pt")

# Correct class names
correct_names = {
    0: "smoke",
    1: "fire",
}

model.model.names = correct_names

# Run prediction
results = model(["/home/hackia_group/Data/Image/F_984.jpg"])

# Process results
for i, result in enumerate(results):
    result.names = correct_names

    boxes = result.boxes

    for box in boxes:
        class_id = int(box.cls[0])
        class_name = correct_names[class_id]
        confidence = float(box.conf[0])

        print(class_name, confidence)

    result.show()
    result.save(filename=f"/home/hackia_group/Data/Results/result_{i}.jpg")

0: 352x640 1 smoke, 52.0ms
Speed: 1.4ms preprocess, 52.0ms inference, 12.1ms postprocess per image at shape (1, 3, 352, 640)
smoke 0.5155274868011475


## Video

In [8]:
import cv2
from ultralytics import YOLO

model = YOLO("/home/hackia_group/Phase1/fire-detection/yolo26_fire_detection7/weights/best.pt")
correct_names = {0: "smoke", 1: "fire"}
model.model.names = correct_names

video_path = "/home/hackia_group/Data/video/Fire_video.mp4"
cap = cv2.VideoCapture(video_path)

# Setup output video
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter("/home/hackia_group/Data/Results/result_video.mp4", fourcc, fps, (w, h))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)
    for result in results:
        result.names = correct_names
        for box in result.boxes:
            class_name = correct_names[int(box.cls[0])]
            confidence = float(box.conf[0])
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 255), 2)
            cv2.putText(frame, f"{class_name} {confidence:.2f}", (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    out.write(frame)

cap.release()
out.release()
print("Vidéo sauvegardée : /Users/youss/Desktop/result_video.mp4")


0: 384x640 (no detections), 42.2ms
Speed: 0.6ms preprocess, 42.2ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 4.8ms
Speed: 0.6ms preprocess, 4.8ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 4.8ms
Speed: 0.6ms preprocess, 4.8ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 4.7ms
Speed: 0.6ms preprocess, 4.7ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 4.6ms
Speed: 0.6ms preprocess, 4.6ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 4.6ms
Speed: 0.6ms preprocess, 4.6ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)



0: 384x640 (no detections), 4.7ms
Speed: 0.5ms preprocess, 4.7ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 4.8ms
Speed: 0.6ms preprocess, 4.8ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 4.6ms
Speed: 0.6ms preprocess, 4.6ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 4.6ms
Speed: 0.6ms preprocess, 4.6ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 4.8ms
Speed: 0.7ms preprocess, 4.8ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 4.7ms
Speed: 0.6ms preprocess, 4.7ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 4.6ms
Speed: 0.6ms preprocess, 4.6ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 4.6ms
Speed: 0.5ms preprocess, 4.6ms inference, 0.3ms p

# GradCam implementation

## Config

In [4]:
WEIGHTS   = "/home/hackia_group/Phase1/fire-detection/yolo26_fire_detection7/weights/best.pt"
VIDEO_IN  = "/home/hackia_group/Data/video/Fire_video.mp4"
VIDEO_OUT = "/home/hackia_group/Data/Results/result_video_gradcam.mp4"
IMG_SIZE  = 640          # Multiple de 32 — obligatoire pour YOLO
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Back up

## Model loading

In [30]:
yolo = YOLO(WEIGHTS)
yolo.model.names = {0: "smoke", 1: "fire"}
yolo.model.eval()
backbone = yolo.model.model   # nn.Sequential des couches YOLO

## Utils 

In [42]:
# ─── Trouver la dernière Conv2d ───────────────────────────────────────────────
def get_last_conv_layer(model):
    last_conv = None
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            last_conv = m
    if last_conv is None:
        raise ValueError("Aucune couche Conv2d trouvée.")
    return last_conv

# ─── Transform : taille divisible par 32 ─────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),   # 640×640 ✓
    transforms.ToTensor(),                      # [0,1]
])

# ─── GradCAM sur le backbone YOLO ─────────────────────────────────────────────
def apply_gradcam_overlay(backbone, tensor_img: torch.Tensor, target_class: int) -> np.ndarray:
    """
    Retourne une image numpy (H,W,3) uint8 avec le GradCAM en overlay.
    tensor_img : (1, 3, H, W) float [0,1], requires_grad=True
    """
    last_conv = get_last_conv_layer(backbone)
    gradcam   = LayerGradCam(backbone, last_conv)

    # GradCAM — on active les gradients ici seulement
    tensor_img = tensor_img.clone().requires_grad_(True)
    attribution = gradcam.attribute(tensor_img, target=target_class, relu_attributions=True)
    # Heatmap → numpy
    heatmap = attribution[0, 0].cpu().detach().numpy()          # (H', W')
    img_np  = tensor_img[0].cpu().permute(1, 2, 0).detach().numpy()  # (H, W, 3)
    H, W    = img_np.shape[:2]
    heatmap = cv2.resize(heatmap, (W, H))
    heatmap = np.maximum(heatmap, 0)
    if heatmap.max() > 0:
        heatmap /= heatmap.max()

    # Colormap jet → BGR uint8
    heatmap_color = cv2.applyColorMap((heatmap * 255).astype(np.uint8), cv2.COLORMAP_JET)
    img_bgr       = cv2.cvtColor((img_np * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)

    overlay = cv2.addWeighted(img_bgr, 0.6, heatmap_color, 0.4, 0)
    return overlay   # BGR uint8, taille (IMG_SIZE, IMG_SIZE, 3)

NameError: name 'IMG_SIZE' is not defined

In [43]:
class YOLOClassifier(nn.Module):
    """
    Extrait les features de la dernière Conv2d via hook forward,
    puis applique GAP + projection linéaire pour obtenir des logits.
    """
    def __init__(self, yolo_model, num_classes: int = 2):
        super().__init__()
        self.yolo_model = yolo_model
        self.num_classes = num_classes
        self._features = None

        # Trouver la dernière Conv2d et y attacher un hook
        self.last_conv = get_last_conv_layer(yolo_model)
        self.last_conv.register_forward_hook(self._hook_fn)

    def _hook_fn(self, module, input, output):
        self._features = output  # (B, C, H, W)

    def forward(self, x):
        # Passage complet dans YOLO (on ignore la sortie détection)
        try:
            _ = self.yolo_model(x)
        except Exception:
            pass  # Certaines versions YOLO lèvent une exception en mode eval partiel

        feats = self._features          # (B, C, H, W) capturé par le hook
        pooled = feats.mean(dim=[2, 3]) # Global Average Pooling → (B, C)

        # Projection vers num_classes (sans paramètres appris — score brut par canal)
        # On prend les `num_classes` premiers canaux comme logits proxy
        logits = pooled[:, :self.num_classes]
        return logits

def apply_gradcam_overlay(classifier: YOLOClassifier,
                          tensor_img: torch.Tensor,
                          target_class: int) -> np.ndarray:
    """
    GradCAM manuel — évite le conflit de hooks avec Captum.
    """
    activations = {}
    gradients   = {}

    # ── Hooks temporaires ──────────────────────────────────────────────────
    def forward_hook(module, input, output):
        activations['feat'] = output                  # (B, C, H', W')

    def backward_hook(module, grad_in, grad_out):
        gradients['feat'] = grad_out[0]               # (B, C, H', W')

    h_fwd = classifier.last_conv.register_forward_hook(forward_hook)
    h_bwd = classifier.last_conv.register_full_backward_hook(backward_hook)

    try:
        # ── Forward ────────────────────────────────────────────────────────
        tensor_img = tensor_img.clone().detach().requires_grad_(True)
        logits = classifier(tensor_img)               # (1, num_classes)

        # ── Backward sur la classe cible ───────────────────────────────────
        classifier.zero_grad()
        score = logits[0, target_class]
        score.backward()

        # ── Calcul GradCAM ─────────────────────────────────────────────────
        grads = gradients['feat']                     # (1, C, H', W')
        acts  = activations['feat']                   # (1, C, H', W')

        # Poids = moyenne spatiale des gradients (GAP)
        weights = grads.mean(dim=[2, 3], keepdim=True)  # (1, C, 1, 1)
        cam = (weights * acts).sum(dim=1, keepdim=False) # (1, H', W')
        cam = cam[0].cpu().detach().numpy()              # (H', W')

        # ReLU + normalisation
        cam = np.maximum(cam, 0)
        if cam.max() > 0:
            cam /= cam.max()

    finally:
        # ── Toujours retirer les hooks ──────────────────────────────────────
        h_fwd.remove()
        h_bwd.remove()

    # ── Overlay ────────────────────────────────────────────────────────────
    img_np  = tensor_img[0].cpu().permute(1, 2, 0).detach().numpy()  # (H, W, 3)
    H, W    = img_np.shape[:2]
    cam_resized    = cv2.resize(cam, (W, H))
    heatmap_color  = cv2.applyColorMap(
        (cam_resized * 255).astype(np.uint8), cv2.COLORMAP_JET
    )
    img_bgr = cv2.cvtColor(
        (np.clip(img_np, 0, 1) * 255).astype(np.uint8), cv2.COLOR_RGB2BGR
    )
    overlay = cv2.addWeighted(img_bgr, 0.6, heatmap_color, 0.4, 0)
    return overlay  # BGR uint8, (H, W, 3)

In [44]:
def preprocess_frame(frame_bgr: np.ndarray, img_size: int = 640) -> torch.Tensor:
    """
    Conversion fiable BGR numpy → tenseur BCHW float [0,1] via OpenCV uniquement.
    Garantit une taille divisible par 32.
    """
    # Arrondir à un multiple de 32 (sécurité)
    size = (img_size // 32) * 32  # 640 → 640, 360 → 352, etc.

    resized   = cv2.resize(frame_bgr, (size, size))          # (size, size, 3) BGR
    rgb       = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)      # RGB
    tensor    = torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0  # (3,H,W)
    return tensor.unsqueeze(0)                                 # (1,3,H,W)


# ─── Pipeline vidéo ──────────────────────────────────────────────────────────
IMG_SIZE = 640

cap    = cv2.VideoCapture(VIDEO_IN)
fps    = cap.get(cv2.CAP_PROP_FPS)
W_orig = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H_orig = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(VIDEO_OUT, fourcc, fps, (W_orig, H_orig))

classifier = YOLOClassifier(backbone, num_classes=2).to(DEVICE)

frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # 1) Préparation — plus de torchvision.transforms ici
    tensor = preprocess_frame(frame, IMG_SIZE).to(DEVICE)

    # Vérification (à commenter après validation)
    assert tensor.shape[2] % 32 == 0 and tensor.shape[3] % 32 == 0, \
        f"Taille invalide : {tensor.shape}"
    assert tensor.max() <= 1.0, f"Valeurs non normalisées : max={tensor.max()}"

    # 2) Inférence YOLO
    with torch.no_grad():
        results = yolo(tensor)

    # Classe dominante
    boxes = results[0].boxes
    if boxes is not None and len(boxes) > 0:
        target_cls = int(boxes.cls[torch.argmax(boxes.conf)].item())
    else:
        target_cls = 0

    # 3) GradCAM
    overlay_bgr = apply_gradcam_overlay(classifier, tensor, target_cls)

    # 4) Resize vers format original
    overlay_bgr = cv2.resize(overlay_bgr, (W_orig, H_orig))

    # 5) Annotation
    label = yolo.model.names[target_cls]
    cv2.putText(overlay_bgr, f"Pred: {label}", (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)

    out.write(overlay_bgr)
    frame_idx += 1
    if frame_idx % 30 == 0:
        print(f"  Frame {frame_idx} — shape tenseur : {tensor.shape}")

cap.release()
out.release()
print(f"\nVidéo sauvegardée : {VIDEO_OUT}")

NameError: name 'VIDEO_IN' is not defined

In [8]:
class YOLOGradCAMWrapper(nn.Module):

    def __init__(self, yolo_model):
        super().__init__()
        self.model = yolo_model

    def forward(self, x, target=None):

        outputs = self.model(x)

        # on force une sortie tensor propre
        if isinstance(outputs, (list, tuple)):
            outputs = outputs[0]

        # cas detection YOLO -> convertir en score simple
        if hasattr(outputs, "boxes"):

            # score = confidence max
            if outputs.boxes is not None and len(outputs.boxes) > 0:
                return outputs.boxes.conf.mean().unsqueeze(0)

            return torch.zeros((1,), device=x.device)

        if torch.is_tensor(outputs):
            return outputs

        raise ValueError(f"Unsupported output: {type(outputs)}")

In [9]:
def get_last_conv_layer(model):
    # YOLO backbone/head réel utilisé au forward
    for name, module in reversed(list(model.named_modules())):
        if isinstance(module, nn.Conv2d):
            # ignorer couches inutilisées
            if "cv2" in name or "cv3" in name:
                print(f"Using layer: {name}")
                return module
    raise ValueError("No suitable Conv2d layer found.")

def visualize_gradcam(model, inputs, labels, device):

    wrapped_model = YOLOGradCAMWrapper(model.model)

    last_conv_layer = get_last_conv_layer(wrapped_model)

    gradcam = LayerGradCam(wrapped_model, last_conv_layer)
    model.model.train()   # nécessaire pour autograd
    # clone obligatoire pour sortir du inference_mode
    inputs_gradcam = inputs.clone().detach().requires_grad_(True)

    attribution = gradcam.attribute(
    inputs_gradcam,
    target=labels,
    relu_attributions=True,
    additional_forward_args=None)
    model.model.eval()

    attribution_np = attribution[0].cpu().detach().numpy()
    inputs_np = inputs[0].cpu().permute(1, 2, 0).detach().numpy()
    height, width = inputs_np.shape[:2]

    heatmap = cv2.resize(attribution_np[0], (width, height))
    heatmap = np.maximum(heatmap, 0)
    heatmap = heatmap / (heatmap.max() + 1e-8)
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
    inputs_np = np.uint8(inputs_np * 255)
    overlay = cv2.addWeighted(inputs_np, 0.6, heatmap, 0.4, 0)

    return overlay

img_size = 640
test_transform = transforms.Compose([
    transforms.ToTensor()])

In [10]:
from ultralytics import YOLO
import torch.nn.functional as F

model = YOLO("/home/hackia_group/Phase1/fire-detection/yolo26_fire_detection7/weights/best.pt")
correct_names = {0: "smoke", 1: "fire"}
model.model.names = correct_names

video_path = "/home/hackia_group/Data/video/Fire_video.mp4"
cap = cv2.VideoCapture(video_path)

# Setup output video
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter("/home/hackia_group/Data/Results/result_video.mp4", fourcc, fps, (w, h))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # model.eval()
    with torch.no_grad():
        frame_resized = cv2.resize(frame, (img_size, img_size))
        frame_tensor = test_transform(frame_resized).unsqueeze(0).to(device)
        #frame = test_transform(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))).unsqueeze(0).to(device)
        results = model(frame_tensor)
        if len(results[0].boxes.cls) > 0:

            label = int(results[0].boxes.cls[0].item())

            gradcam_frame = visualize_gradcam(
                model=model,
                inputs=frame_tensor,
                labels=label,
                device=device
            )

            gradcam_frame = cv2.resize(gradcam_frame, (w, h))

            out.write(gradcam_frame)
        else:
            out.write(frame)

cap.release()
out.release()
print("Vidéo sauvegardée : /Users/youss/Desktop/result_video.mp4")


0: 640x640 (no detections), 4.8ms
Speed: 0.0ms preprocess, 4.8ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 5.1ms
Speed: 0.0ms preprocess, 5.1ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)



0: 640x640 (no detections), 7.3ms
Speed: 0.0ms preprocess, 7.3ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 5.1ms
Speed: 0.0ms preprocess, 5.1ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 5.5ms
Speed: 0.0ms preprocess, 5.5ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 5.1ms
Speed: 0.0ms preprocess, 5.1ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 5.1ms
Speed: 0.0ms preprocess, 5.1ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 5.1ms
Speed: 0.0ms preprocess, 5.1ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 5.3ms
Speed: 0.0ms preprocess, 5.3ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 5.4ms
Speed: 0.0ms preprocess, 5.4ms inference, 0.6ms p

ValueError: Unsupported output: <class 'dict'>

In [12]:
import cv2
import numpy as np
import torch
import torch.nn as nn
from torchvision import transforms
from ultralytics import YOLO

# ─── Config ───────────────────────────────────────────────────────────────────
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
img_size  = 640
VIDEO_IN  = "/home/hackia_group/Data/video/Fire_video.mp4"
VIDEO_OUT = "/home/hackia_group/Data/Results/result_video.mp4"
MODEL_PT  = "/home/hackia_group/Phase1/fire-detection/yolo26_fire_detection7/weights/best.pt"

# ─── Chargement YOLO ──────────────────────────────────────────────────────────
model = YOLO(MODEL_PT)
model.model.names = {0: "smoke", 1: "fire"}
model.model.eval()

# ─── Trouver la dernière Conv2d utile ─────────────────────────────────────────
def get_last_conv_layer(yolo_nn: nn.Module) -> nn.Module:
    for name, module in reversed(list(yolo_nn.named_modules())):
        if isinstance(module, nn.Conv2d) and ("cv2" in name or "cv3" in name):
            print(f"[GradCAM] Couche cible : {name}")
            return module
    raise ValueError("Aucune Conv2d cv2/cv3 trouvée.")

# ─── Wrapper YOLO → logits scalaires ──────────────────────────────────────────
class YOLOGradCAMWrapper(nn.Module):
    """
    Fait passer x dans YOLO et retourne un vecteur (1, num_classes)
    construit depuis les features de la dernière Conv2d via GAP.
    La sortie dict/objet de YOLO est totalement ignorée.
    """
    def __init__(self, yolo_nn: nn.Module, num_classes: int = 2):
        super().__init__()
        self.yolo_nn     = yolo_nn
        self.num_classes = num_classes
        self._features   = None
        self.last_conv   = get_last_conv_layer(yolo_nn)
        # Hook permanent pour capturer les activations
        self.last_conv.register_forward_hook(self._hook_fn)

    def _hook_fn(self, module, input, output):
        self._features = output  # (B, C, H', W')

    def forward(self, x):
        self.yolo_nn(x)          # sortie ignorée (dict / objet YOLO)
        pooled = self._features.mean(dim=[2, 3])        # (B, C)
        return pooled[:, :self.num_classes]             # (B, num_classes)

# ─── GradCAM manuel ───────────────────────────────────────────────────────────
def apply_gradcam_overlay(wrapper: YOLOGradCAMWrapper,
                          tensor_img: torch.Tensor,
                          target_class: int) -> np.ndarray:
    """
    Activation CAM (sans backward) — robuste même si YOLO détache ses tenseurs.
    """
    activations = {}

    h_fwd = wrapper.last_conv.register_forward_hook(
        lambda m, i, o: activations.update({'feat': o.detach()})
    )

    try:
        with torch.no_grad():
            wrapper.eval()
            wrapper(tensor_img)

        acts = activations['feat']          # (1, C, H', W')

        # CAM = moyenne sur les canaux (pas besoin de gradients)
        cam = acts[0].mean(dim=0).cpu().numpy()   # (H', W')

        cam = np.maximum(cam, 0)
        if cam.max() > 0:
            cam /= cam.max()

    finally:
        h_fwd.remove()

    # ── Overlay ───────────────────────────────────────────────────────────
    img_np = tensor_img[0].cpu().permute(1, 2, 0).numpy()   # (H, W, 3)
    H, W   = img_np.shape[:2]

    cam_up        = cv2.resize(cam, (W, H))
    heatmap_color = cv2.applyColorMap(
        (cam_up * 255).astype(np.uint8), cv2.COLORMAP_JET
    )
    img_bgr = cv2.cvtColor(
        (np.clip(img_np, 0, 1) * 255).astype(np.uint8), cv2.COLOR_RGB2BGR
    )
    return cv2.addWeighted(img_bgr, 0.6, heatmap_color, 0.4, 0)


# ─── Prétraitement ────────────────────────────────────────────────────────────
def preprocess_frame(frame_bgr: np.ndarray, size: int = 640) -> torch.Tensor:
    size    = (size // 32) * 32
    resized = cv2.resize(frame_bgr, (size, size))
    rgb     = cv2.cvtColor(resized, cv2.COLOR_BGR2RGB)
    tensor  = torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0
    return tensor.unsqueeze(0)                         # (1, 3, H, W)

# ─── Initialisation wrapper (une seule fois) ──────────────────────────────────
wrapper = YOLOGradCAMWrapper(model.model, num_classes=2).to(device)

# ─── Pipeline vidéo ───────────────────────────────────────────────────────────
cap    = cv2.VideoCapture(VIDEO_IN)
fps    = cap.get(cv2.CAP_PROP_FPS)
W_orig = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H_orig = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out    = cv2.VideoWriter(VIDEO_OUT, fourcc, fps, (W_orig, H_orig))

frame_idx = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # 1) Prétraitement
    tensor = preprocess_frame(frame, img_size).to(device)

    # 2) Inférence YOLO (pour récupérer la classe détectée)
    with torch.no_grad():
        results = model(tensor)

    boxes = results[0].boxes
    if boxes is not None and len(boxes) > 0:
        # Classe avec la confidence la plus haute
        target_cls = int(boxes.cls[torch.argmax(boxes.conf)].item())

        # 3) GradCAM
        overlay = apply_gradcam_overlay(wrapper, tensor, target_cls)

        # 4) Annotation
        label = model.model.names[target_cls]
        cv2.putText(overlay, f"Pred: {label}", (20, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)

        # 5) Resize vers résolution originale
        overlay = cv2.resize(overlay, (W_orig, H_orig))
        out.write(overlay)
    else:
        # Aucune détection → frame originale
        out.write(frame)

    frame_idx += 1
    if frame_idx % 30 == 0:
        print(f"  Frame {frame_idx} traitée")

cap.release()
out.release()
print(f"Vidéo sauvegardée : {VIDEO_OUT}")

[GradCAM] Couche cible : model.23.cv3.2.2

0: 640x640 (no detections), 4.5ms
Speed: 0.0ms preprocess, 4.5ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 5.4ms
Speed: 0.0ms preprocess, 5.4ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 5.4ms
Speed: 0.0ms preprocess, 5.4ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 5.3ms
Speed: 0.0ms preprocess, 5.3ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 5.1ms
Speed: 0.0ms preprocess, 5.1ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 5.3ms
Speed: 0.0ms preprocess, 5.3ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 5.2ms
Speed: 0.0ms preprocess, 5.2ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 (no detections), 5.1ms
Speed:

In [ ]:
import os
import cv2
import numpy as np
import torch
import quantus
from ultralytics import YOLO

MODEL_PATH = "/Users/youss/Desktop/fire-detection/yolo26_fire_detection7/weights/best.pt"
IMAGES_DIR = "/Users/youss/Desktop/D-Fire/test/images"
LABELS_DIR = "/Users/youss/Desktop/D-Fire/test/labels"
IMG_SIZE = 640

model = YOLO(MODEL_PATH)
torch_model = model.model
torch_model.train()

def load_image(path):
    img = cv2.imread(path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def yolo_label_to_mask(label_path, img_h, img_w):
    mask = np.zeros((img_h, img_w), dtype=np.float32)
    if not os.path.exists(label_path) or os.path.getsize(label_path) == 0:
        return mask
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            _, cx, cy, w, h = map(float, parts)
            x1 = int((cx - w/2) * img_w)
            y1 = int((cy - h/2) * img_h)
            x2 = int((cx + w/2) * img_w)
            y2 = int((cy + h/2) * img_h)
            mask[y1:y2, x1:x2] = 1.0
    return mask

target_layer = torch_model.model[9]

def get_gradcam_attribution(img_rgb, target_layer):
    img_resized = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
    tensor = torch.from_numpy(img_resized).permute(2,0,1).float().unsqueeze(0) / 255.0

    activations = []
    gradients = []

    def forward_hook(module, input, output):
        activations.append(output.detach())

    def backward_hook(module, grad_in, grad_out):
        gradients.append(grad_out[0].detach())

    h1 = target_layer.register_forward_hook(forward_hook)
    h2 = target_layer.register_full_backward_hook(backward_hook)

    with torch.enable_grad():
        tensor = tensor.requires_grad_(True)
        output = torch_model(tensor)
        if isinstance(output, (tuple, list)):
            output = output[0]
        if isinstance(output, dict):
            output = list(output.values())[0]
        loss = output.max()
        loss.backward()

    h1.remove()
    h2.remove()

    act = activations[0].squeeze(0)
    grad = gradients[0].squeeze(0)
    weights = grad.mean(dim=(1, 2))
    cam = (weights[:, None, None] * act).sum(0)
    cam = torch.relu(cam).numpy()
    cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam

img_files_all = set(f.replace('.jpg','') for f in os.listdir(IMAGES_DIR) if f.endswith('.jpg'))
lbl_files_all = set(f.replace('.txt','') for f in os.listdir(LABELS_DIR))
common = img_files_all & lbl_files_all
img_files = [f + '.jpg' for f in common][:500]

images_list, attributions_list, masks_list = [], [], []

for fname in img_files:
    img_path = os.path.join(IMAGES_DIR, fname)
    label_path = os.path.join(LABELS_DIR, fname.replace('.jpg', '.txt'))

    img = load_image(img_path)
    if img is None:
        continue
    mask = yolo_label_to_mask(label_path, IMG_SIZE, IMG_SIZE)
    if mask.sum() == 0:
        continue

    attribution = get_gradcam_attribution(img, target_layer)
    img_resized = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    images_list.append(img_resized.transpose(2,0,1) / 255.0)
    attributions_list.append(attribution[np.newaxis])
    masks_list.append(mask[np.newaxis])

x_batch = np.array(images_list)
a_batch = np.array(attributions_list)
s_batch = np.array(masks_list).astype(bool)

print(f"x_batch: {x_batch.shape}")
print(f"a_batch: {a_batch.shape}")
print(f"s_batch: {s_batch.shape}")
print(f"Images avec annotations: {len(x_batch)}")

pointing_game = quantus.PointingGame(normalise=True, abs=True)
attr_loc = quantus.AttributionLocalisation(normalise=True, abs=True)
top_k = quantus.TopKIntersection(normalise=True, abs=True)

y_batch = np.zeros(len(x_batch), dtype=int)

pg_scores = pointing_game(model=None, x_batch=x_batch, y_batch=y_batch, a_batch=a_batch, s_batch=s_batch, channel_first=True)
al_scores = attr_loc(model=None, x_batch=x_batch, y_batch=y_batch, a_batch=a_batch, s_batch=s_batch, channel_first=True)
tk_scores = top_k(model=None, x_batch=x_batch, y_batch=y_batch, a_batch=a_batch, s_batch=s_batch, channel_first=True)

print(f"Pointing Game:            {np.mean(pg_scores):.3f}")
print(f"Attribution Localisation: {np.mean(al_scores):.3f}")
print(f"Top-K Intersection:       {np.mean(tk_scores):.3f}")

/var/folders/zm/t4ysddjs4ln1sxxcd0rn26t40000gn/T/ipykernel_16059/3922909985.py:67: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()


x_batch: (18, 3, 640, 640)
a_batch: (18, 1, 640, 640)
s_batch: (18, 1, 640, 640)
Images avec annotations: 18


AssertionError: The elements in the attribution vector are all equal to zero, which may cause inconsistent results since many metrics rely on ordering. Recompute the explanations.

In [25]:
img_files = set(f.replace('.jpg','') for f in os.listdir(IMAGES_DIR) if f.endswith('.jpg'))
lbl_files = set(f.replace('.txt','') for f in os.listdir(LABELS_DIR))
common = img_files & lbl_files
print(f"Correspondances: {len(common)}")

Correspondances: 4306
